# 🔗 Data Fusion: Social Media Influence on Online Purchase Behavior
### A Production-Ready Data Pipeline
---
**Datasets**
| # | File | Description | Size |
|---|------|-------------|------|
| 1 | `amz_br_total_products_data_processed.csv` | Amazon Brazil product catalog | ~1.34 M rows |
| 2 | `online_shoppers.csv` | E-commerce session behavior | ~1.00 M rows |
| 3 | `twitter_sentiment_dataset.csv` | Social media sentiment & engagement | ~503 K rows |

**Pipeline Stages**: Load → Clean → Feature Engineering → Fusion → ML → Evaluate → Visualize


## 📦 Section 0 – Imports & Configuration

In [ ]:
# ── Standard Library ───────────────────────────────────────────────────────
import os, warnings, re, sqlite3
from pathlib import Path

# ── Data Manipulation ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ──────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine Learning ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Plot Style ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#e6edf3",
    "axes.titlecolor":  "#e6edf3",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#e6edf3",
    "grid.color":       "#21262d",
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "figure.dpi":       120,
})
PALETTE = ["#58a6ff", "#3fb950", "#f78166", "#d2a8ff", "#ffa657", "#79c0ff"]

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path("dataraw")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

PRODUCT_FILE   = DATA_DIR / "amz_br_total_products_data_processed.csv"
ECOMMERCE_FILE = DATA_DIR / "online_shoppers.csv"
TWITTER_FILE   = DATA_DIR / "twitter_sentiment_dataset.csv"

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All libraries imported.")
print(f"   Output directory: {OUTPUT_DIR.resolve()}")


---
## 📂 Section 1 – `load_data()`: Load & Inspect All Datasets

We load the three datasets and print `.info()` / `.head()` dynamically so
the pipeline adapts to any column changes without manual editing.


In [ ]:
def load_data(
    product_file=PRODUCT_FILE,
    ecommerce_file=ECOMMERCE_FILE,
    twitter_file=TWITTER_FILE,
    product_sample: int | None = 300_000,
    ecommerce_sample: int | None = 300_000,
    twitter_sample: int | None = 200_000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load the three raw CSVs into DataFrames.

    Parameters
    ----------
    *_sample : int or None
        If given, reads only this many rows for speed.
        Set to None to load the full file.

    Returns
    -------
    (df_product, df_ecommerce, df_twitter) – raw unprocessed DataFrames
    """
    print("=" * 65)
    print("  LOAD DATA")
    print("=" * 65)

    # ── 1. Product ──────────────────────────────────────────────────
    print("\n📦 [1/3] Loading Product dataset …")
    df_product = pd.read_csv(product_file, nrows=product_sample, low_memory=False)
    print(f"   Shape : {df_product.shape}")
    print(f"   Columns: {df_product.columns.tolist()}")
    display(df_product.head(3))
    df_product.info(memory_usage="deep")

    # ── 2. Ecommerce ────────────────────────────────────────────────
    print("\n🛒 [2/3] Loading Ecommerce dataset …")
    df_ecommerce = pd.read_csv(ecommerce_file, nrows=ecommerce_sample, low_memory=False)
    print(f"   Shape : {df_ecommerce.shape}")
    print(f"   Columns: {df_ecommerce.columns.tolist()}")
    display(df_ecommerce.head(3))
    df_ecommerce.info(memory_usage="deep")

    # ── 3. Twitter ──────────────────────────────────────────────────
    print("\n🐦 [3/3] Loading Twitter Sentiment dataset …")
    df_twitter = pd.read_csv(twitter_file, nrows=twitter_sample, low_memory=False)
    print(f"   Shape : {df_twitter.shape}")
    print(f"   Columns: {df_twitter.columns.tolist()}")
    display(df_twitter.head(3))
    df_twitter.info(memory_usage="deep")

    print("\n✅ All datasets loaded successfully.")
    return df_product, df_ecommerce, df_twitter


df_product_raw, df_ecommerce_raw, df_twitter_raw = load_data()


---
## 🧹 Section 2 – Cleaning Functions

### 2A – `clean_product(df)`


In [ ]:
def clean_product(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the Amazon product catalog.

    Steps
    -----
    - Drop duplicates on ASIN
    - Handle missing / zero prices
    - Coerce numeric columns
    - Normalise category name encoding
    - Create price_discount feature
    """
    print("\n🔧 Cleaning Product dataset …")
    df = df.copy()

    # --- Drop complete duplicates ---
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f"   Duplicates removed  : {before - len(df):,}")

    # --- Drop duplicates by ASIN (keep first) ---
    if "asin" in df.columns:
        before = len(df)
        df.drop_duplicates(subset="asin", keep="first", inplace=True)
        print(f"   Duplicate ASINs     : {before - len(df):,}")

    # --- Coerce numeric types dynamically ---
    numeric_candidates = ["stars", "reviews", "price", "listPrice", "boughtInLastMonth"]
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Fix zero/negative prices ---
    if "price" in df.columns:
        invalid_price = df["price"].isna() | (df["price"] <= 0)
        print(f"   Invalid prices      : {invalid_price.sum():,} (will fill with median)")
        price_median = df.loc[~invalid_price, "price"].median()
        df.loc[invalid_price, "price"] = price_median

    # --- listPrice: treat 0 as NaN, fill with price ---
    if "listPrice" in df.columns and "price" in df.columns:
        df["listPrice"] = df["listPrice"].replace(0, np.nan)
        df["listPrice"].fillna(df["price"], inplace=True)

    # --- Derived: discount percentage ---
    if "listPrice" in df.columns and "price" in df.columns:
        df["price_discount_pct"] = ((df["listPrice"] - df["price"]) / df["listPrice"]).clip(0, 1)

    # --- Boolean cast ─────────────────────────────────────────────
    if "isBestSeller" in df.columns:
        df["isBestSeller"] = df["isBestSeller"].map({"True": 1, "False": 0, True: 1, False: 0}).fillna(0).astype(int)

    # --- Clean category names (strip whitespace, normalise accented) ---
    if "categoryName" in df.columns:
        df["categoryName"] = df["categoryName"].str.strip()

    # --- Drop URL columns (not needed for analysis) ------------------
    url_cols = [c for c in df.columns if "url" in c.lower() or "img" in c.lower()]
    df.drop(columns=url_cols, errors="ignore", inplace=True)

    # --- Missing value report ---
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print("   Remaining missing values:")
        print(missing.to_string(header=False))
        df.dropna(subset=["categoryName"], inplace=True)

    print(f"   ✅ Product clean shape: {df.shape}")
    return df.reset_index(drop=True)


df_product = clean_product(df_product_raw)
display(df_product.head(3))


### 2B – `clean_ecommerce(df)`

In [ ]:
def clean_ecommerce(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the online shoppers session dataset.

    Steps
    -----
    - Detect & cast Revenue to binary int
    - Handle missing numeric values with median imputation
    - Remove impossible values (negative durations)
    - Encode Month to integer (1-12)
    - Encode VisitorType & Weekend
    """
    print("\n🔧 Cleaning Ecommerce dataset …")
    df = df.copy()

    # --- Drop full duplicates ---
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f"   Duplicates removed  : {before - len(df):,}")

    # --- Detect Revenue column dynamically ---
    rev_col = None
    for c in df.columns:
        if c.lower() == "revenue":
            rev_col = c
            break
    if rev_col is None:
        raise ValueError("No 'Revenue' column found in ecommerce dataset!")
    print(f"   Revenue column      : '{rev_col}'")
    df["Revenue"] = df[rev_col].map({True: 1, False: 0, "True": 1, "False": 0}).fillna(0).astype(int)
    print(f"   Revenue distribution:\n{df['Revenue'].value_counts().to_string()}")

    # --- Month encoding ---
    month_map = {
        "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "June": 6,
        "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
    }
    if "Month" in df.columns:
        df["Month_num"] = df["Month"].map(month_map)
        missing_months = df["Month_num"].isna().sum()
        if missing_months > 0:
            print(f"   ⚠ Unknown month values: {missing_months} – filling with mode")
            df["Month_num"].fillna(df["Month_num"].mode()[0], inplace=True)

    # --- VisitorType ---
    if "VisitorType" in df.columns:
        vt_map = {"Returning_Visitor": 0, "New_Visitor": 1, "Other": 2}
        df["VisitorType_enc"] = df["VisitorType"].map(vt_map).fillna(2).astype(int)

    # --- Weekend ---
    if "Weekend" in df.columns:
        df["Weekend"] = df["Weekend"].map({True: 1, False: 0, "True": 1, "False": 0}).fillna(0).astype(int)

    # --- Coerce all duration/rate columns to numeric ---
    numeric_cols = [
        c for c in df.columns
        if any(kw in c for kw in ["Duration", "Rates", "Values", "Related", "Administrative",
                                   "Informational", "SpecialDay"])
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Remove impossible negatives ---
    duration_cols = [c for c in df.columns if "Duration" in c]
    for col in duration_cols:
        df.loc[df[col] < 0, col] = np.nan

    # --- Median imputation for remaining NaNs ---
    numeric_df_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_df_cols:
        if df[col].isna().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)

    print(f"   ✅ Ecommerce clean shape: {df.shape}")
    return df.reset_index(drop=True)


df_ecommerce = clean_ecommerce(df_ecommerce_raw)
display(df_ecommerce.head(3))


### 2C – `clean_social(df)`

In [ ]:
def clean_social(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the Twitter sentiment dataset.

    Steps
    -----
    - Parse timestamps
    - Coerce engagement counts to numeric
    - Handle missing sentiment labels
    - Detect & normalise sentiment_score to [-1, 1]
    - Drop duplicates by tweet id
    """
    print("\n🔧 Cleaning Twitter/Social dataset …")
    df = df.copy()

    # --- Timestamp parsing ---
    if "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
        invalid_ts = df["created_at"].isna().sum()
        if invalid_ts > 0:
            print(f"   ⚠ Invalid timestamps: {invalid_ts} – dropping rows")
            df.dropna(subset=["created_at"], inplace=True)
        df["tweet_month"] = df["created_at"].dt.month
        df["tweet_year"]  = df["created_at"].dt.year
        df["tweet_dow"]   = df["created_at"].dt.dayofweek   # 0=Mon, 6=Sun

    # --- Drop duplicate tweet IDs ---
    if "id" in df.columns:
        before = len(df)
        df.drop_duplicates(subset="id", keep="first", inplace=True)
        print(f"   Duplicate tweets removed : {before - len(df):,}")

    # --- Coerce engagement metrics ---
    engagement_cols = ["retweet_count", "like_count", "reply_count",
                       "quote_count", "impression_count",
                       "user_followers_count", "user_following_count"]
    for col in engagement_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0)

    # --- Sentiment label normalisation ---
    if "sentiment" in df.columns:
        df["sentiment"] = df["sentiment"].str.strip().str.lower()
        valid_sentiments = {"positive", "neutral", "negative"}
        unknown_mask = ~df["sentiment"].isin(valid_sentiments)
        if unknown_mask.sum() > 0:
            print(f"   ⚠ Unknown sentiments: {unknown_mask.sum()} → mapping to 'neutral'")
            df.loc[unknown_mask, "sentiment"] = "neutral"

    # --- sentiment_score normalisation ---
    if "sentiment_score" in df.columns:
        df["sentiment_score"] = pd.to_numeric(df["sentiment_score"], errors="coerce")
        # Detect if score is 0–1 style (no negatives) → re-centre around 0
        if df["sentiment_score"].min() >= 0:
            df["sentiment_score"] = 2 * df["sentiment_score"] - 1  # remap to [-1, 1]
        df["sentiment_score"] = df["sentiment_score"].clip(-1, 1)
        df["sentiment_score"].fillna(0, inplace=True)

    # --- Filter to English tweets if language column exists ---
    if "language" in df.columns:
        lang_before = len(df)
        df = df[df["language"] == "en"].copy()
        print(f"   Non-English rows removed: {lang_before - len(df):,}")

    # --- Drop heavy text columns not needed downstream ---
    drop_cols = [c for c in ["text", "urls", "media_urls", "hashtags",
                               "mentions", "source", "attack_type",
                               "delivery_method", "context_target",
                               "in_reply_to_user_id", "conversation_id"] if c in df.columns]
    df.drop(columns=drop_cols, errors="ignore", inplace=True)

    print(f"   ✅ Social clean shape: {df.shape}")
    print(f"   Sentiment distribution:\n{df['sentiment'].value_counts().to_string()}")
    return df.reset_index(drop=True)


df_twitter = clean_social(df_twitter_raw)
display(df_twitter.head(3))


---
## ⚙️ Section 3 – Feature Engineering

### 3A – `feature_engineering_product(df)` — Product Features


In [ ]:
# ── Broad category mapping (Portuguese→English label) ──────────────────────
CATEGORY_MAP = {
    # Electronics
    "Eletrônicos": "Electronics",
    "Acessórios e Artigos Eletrônicos": "Electronics",
    "Eletrônicos e Aparelhos": "Electronics",
    "Celulares e Comunicação": "Electronics",
    "Celulares e Smartphones": "Electronics",
    "Computadores e Informática": "Electronics",
    "Notebooks": "Electronics",
    "Tablets": "Electronics",
    "TVs": "Electronics",
    "TV, Áudio e Cinema em Casa": "Electronics",
    "Fones de Ouvido": "Electronics",
    "Wearables e Tecnologia Vestível": "Electronics",
    "Casa Inteligente": "Electronics",
    "Câmeras Digitais": "Electronics",
    "Games": "Gaming",
    "PlayStation 4, Jogos, Consoles e Acessórios": "Gaming",
    "PlayStation 5, Jogos, Consoles e Acessórios": "Gaming",
    "Nintendo Switch, Jogos, Consoles e Acessórios": "Gaming",
    "Xbox One, Jogos, Consoles e Acessórios": "Gaming",
    "Xbox Series X e S, Jogos, Consoles e Acessórios": "Gaming",
    "Livros": "Books",
    "eBooks Kindle": "Books",
    "Livros Infantis": "Books",
    "Livros Universitários, Técnicos e Profissionais": "Books",
    "Roupas, Calçados e Joias": "Fashion",
    "Feminino": "Fashion",
    "Masculino": "Fashion",
    "Meninas": "Fashion",
    "Meninos": "Fashion",
    "Loja de Calçados": "Fashion",
    "Bolsas": "Fashion",
    "Beleza": "Beauty & Health",
    "Maquiagem": "Beauty & Health",
    "Saúde e Cuidados Pessoais": "Beauty & Health",
    "Vitaminas, Minerais e Suplementos": "Beauty & Health",
    "Perfumes e Fragrâncias": "Beauty & Health",
    "Produtos de Cuidados com a Pele": "Beauty & Health",
    "Cuidados com o Cabelo": "Beauty & Health",
    "Casa": "Home & Garden",
    "Cozinha": "Home & Garden",
    "Móveis e Decoração": "Home & Garden",
    "Organização e Armazenamento para Casa": "Home & Garden",
    "Jardim e Piscina": "Home & Garden",
    "Eletrodomésticos": "Home & Garden",
    "Iluminação": "Home & Garden",
    "Pet Shop": "Pet Supplies",
    "Produtos para Cães": "Pet Supplies",
    "Acessórios para Gatos": "Pet Supplies",
    "Esportes e Aventura": "Sports",
    "Equipamento para Exercícios e Academia": "Sports",
    "Material de Futebol": "Sports",
    "Alimentos e Bebidas": "Food & Drinks",
    "Bebidas Alcoólicas": "Food & Drinks",
    "Lanches e Doces": "Food & Drinks",
    "Brinquedos e Jogos": "Toys",
    "Jogos de Tabuleiro": "Toys",
    "Automotivo": "Automotive",
    "Ferramentas e Materiais de Construção": "Tools & DIY",
    "Ferramentas Elétricas": "Tools & DIY",
}


def feature_engineering_product(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived features to the product DataFrame.
    """
    print("\n⚙ Feature engineering – Product …")
    df = df.copy()

    # ── Broad category ────────────────────────────────────────────
    df["category_broad"] = df["categoryName"].map(CATEGORY_MAP).fillna("Other")

    # ── Price tiers ───────────────────────────────────────────────
    price_bins  = [0, 50, 150, 500, 1500, np.inf]
    price_labels = ["budget", "low", "mid", "high", "premium"]
    df["price_tier"] = pd.cut(df["price"], bins=price_bins, labels=price_labels, right=True)

    # ── Popularity score = normalised (stars × log(reviews+1)) ───
    df["log_reviews"] = np.log1p(df["reviews"].fillna(0))
    df["popularity_score"] = df["stars"].fillna(df["stars"].median()) * df["log_reviews"]

    # ── Demand signal = log(boughtInLastMonth+1) ─────────────────
    if "boughtInLastMonth" in df.columns:
        df["demand_signal"] = np.log1p(df["boughtInLastMonth"].fillna(0))

    # ── Best-seller premium (prices compared per category) ────────
    cat_median_price = df.groupby("category_broad")["price"].transform("median")
    df["price_vs_cat_median"] = df["price"] / cat_median_price.replace(0, np.nan)

    print(f"   category_broad distribution:\n{df['category_broad'].value_counts().head(10).to_string()}")
    print(f"   ✅ Product feature shape: {df.shape}")
    return df


df_product = feature_engineering_product(df_product)
display(df_product[["title", "price", "stars", "category_broad", "price_tier",
                     "popularity_score"]].head(5))


### 3B – `aggregate_social(df)` — Social Media Aggregation

In [ ]:
def aggregate_social(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate Twitter data by calendar month to produce a monthly
    social-sentiment feature vector. This is later joined to the
    ecommerce dataset (which also has Month).

    Returns a DataFrame indexed by month (1-12) with:
      avg_sentiment_score, pct_positive, pct_negative,
      avg_engagement, avg_subjectivity, sentiment_volatility
    """
    print("\n⚙ Aggregating Social Media data …")
    df = df.copy()

    # ── Numeric sentiment label ───────────────────────────────────
    sentiment_num = {"positive": 1, "neutral": 0, "negative": -1}
    df["sentiment_num"] = df["sentiment"].map(sentiment_num).fillna(0)

    # ── Engagement = likes + retweets ────────────────────────────
    df["engagement"] = df.get("like_count", 0) + df.get("retweet_count", 0)

    # ── Aggregate by tweet_month ──────────────────────────────────
    if "tweet_month" not in df.columns:
        raise ValueError("tweet_month column missing – run clean_social() first.")

    agg = df.groupby("tweet_month").agg(
        avg_sentiment_score = ("sentiment_score", "mean"),
        std_sentiment_score = ("sentiment_score", "std"),      # volatility
        pct_positive        = ("sentiment_num",   lambda x: (x == 1).mean()),
        pct_negative        = ("sentiment_num",   lambda x: (x == -1).mean()),
        avg_engagement      = ("engagement",       "mean"),
        total_tweets        = ("sentiment_num",    "count"),
    ).reset_index()

    if "subjectivity" in df.columns:
        sub_agg = df.groupby("tweet_month")["subjectivity"].mean().reset_index()
        sub_agg.columns = ["tweet_month", "avg_subjectivity"]
        agg = agg.merge(sub_agg, on="tweet_month", how="left")

    agg.rename(columns={"tweet_month": "Month_num"}, inplace=True)
    agg["std_sentiment_score"].fillna(0, inplace=True)

    print(f"   ✅ Social aggregate shape: {agg.shape}")
    display(agg)
    return agg


df_social_agg = aggregate_social(df_twitter)


### 3C – `feature_engineering_ecommerce(df)` — Session Features

In [ ]:
def feature_engineering_ecommerce(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive behavioural session features from the ecommerce dataset.
    """
    print("\n⚙ Feature engineering – Ecommerce …")
    df = df.copy()

    # ── Total pages visited ───────────────────────────────────────
    page_cols = [c for c in df.columns if c in ["Administrative", "Informational", "ProductRelated"]]
    if page_cols:
        df["total_pages"] = df[page_cols].sum(axis=1)

    # ── Total time on site ────────────────────────────────────────
    dur_cols = [c for c in df.columns if "Duration" in c]
    if dur_cols:
        df["total_duration_sec"] = df[dur_cols].sum(axis=1)

    # ── Product engagement ratio ──────────────────────────────────
    if all(c in df.columns for c in ["ProductRelated", "total_pages"]):
        df["product_page_ratio"] = np.where(
            df["total_pages"] > 0,
            df["ProductRelated"] / df["total_pages"],
            0,
        )

    # ── Log-transform skewed numeric columns ──────────────────────
    skew_cols = ["ProductRelated_Duration", "total_duration_sec"]
    for col in skew_cols:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))

    # ── PageValues bucketing (key predictor) ─────────────────────
    if "PageValues" in df.columns:
        df["has_page_value"] = (df["PageValues"] > 0).astype(int)
        df["log_PageValues"] = np.log1p(df["PageValues"].clip(lower=0))

    print(f"   ✅ Ecommerce feature shape: {df.shape}")
    return df


df_ecommerce = feature_engineering_ecommerce(df_ecommerce)
display(df_ecommerce.head(3))


---
## 🔗 Section 4 – `merge_data()`: Data Fusion

**Fusion strategy** (cross-dataset without data leakage):

| Join | Key | Type |
|------|-----|------|
| Ecommerce ← Social aggregate | `Month_num` | left join |
| Fusion ← Product aggregate | (global stats) | broadcast |

Because the ecommerce sessions have no product ID, we enrich each session
with the **monthly social climate** (global sentiment that month) and the
**overall product catalog statistics** (category-level price benchmarks).


In [ ]:
def _aggregate_product_globally(df_prod: pd.DataFrame) -> dict:
    """
    Compute category-level product aggregates to broadcast onto sessions.
    Returns a dict of overall stats + a category-level Price/Stars table.
    """
    global_stats = {
        "global_avg_price"       : df_prod["price"].mean(),
        "global_avg_stars"       : df_prod["stars"].mean(),
        "global_avg_reviews"     : df_prod["reviews"].mean(),
        "global_avg_discount_pct": df_prod["price_discount_pct"].mean() if "price_discount_pct" in df_prod.columns else 0,
        "global_bestseller_rate" : df_prod["isBestSeller"].mean(),
        "global_demand_signal"   : df_prod["demand_signal"].mean() if "demand_signal" in df_prod.columns else 0,
    }
    return global_stats


def merge_data(
    df_ecommerce: pd.DataFrame,
    df_social_agg: pd.DataFrame,
    df_product: pd.DataFrame,
) -> pd.DataFrame:
    """
    Fuse all three datasets into a single modelling-ready DataFrame.

    Parameters
    ----------
    df_ecommerce   : cleaned + feature-engineered ecommerce sessions
    df_social_agg  : monthly aggregated social sentiment
    df_product     : cleaned + feature-engineered product catalog

    Returns
    -------
    df_fused : unified DataFrame, no data leakage, ready for ML
    """
    print("\n" + "=" * 65)
    print("  DATA FUSION")
    print("=" * 65)

    df = df_ecommerce.copy()

    # ── Step 1: Join monthly social sentiment ─────────────────────
    if "Month_num" not in df.columns:
        raise ValueError("Ecommerce must have 'Month_num' column after cleaning.")

    print(f"\n1️⃣  Joining social sentiment aggregate on Month_num …")
    df = df.merge(df_social_agg, on="Month_num", how="left")
    print(f"   Shape after social join : {df.shape}")

    # Fill any missing month sentiment (months not in twitter data)
    social_cols = [c for c in df_social_agg.columns if c != "Month_num"]
    for col in social_cols:
        if col in df.columns:
            df[col].fillna(df[col].median(), inplace=True)

    # ── Step 2: Broadcast global product stats ────────────────────
    print(f"\n2️⃣  Broadcasting global product statistics …")
    global_stats = _aggregate_product_globally(df_product)
    for k, v in global_stats.items():
        df[k] = v
    print(f"   Product stats added     : {list(global_stats.keys())}")

    # ── Step 3: Category-level price signal (top categories) ──────
    print(f"\n3️⃣  Adding category price tier distribution …")
    cat_price = df_product.groupby("category_broad")["price"].agg(["mean", "std"]).reset_index()
    cat_price.columns = ["category_broad", "cat_avg_price", "cat_price_std"]
    # Encode as the ratio Electronics/overall for a single signal
    electronics_avg = cat_price.loc[cat_price["category_broad"] == "Electronics", "cat_avg_price"]
    df["prod_electronics_price_ref"] = float(electronics_avg.values[0]) if len(electronics_avg) > 0 else global_stats["global_avg_price"]

    # ── Step 4: Composite fusion features ────────────────────────
    print(f"\n4️⃣  Creating composite fusion features …")
    # Sentiment × PageValue interaction (social buzz × user intent)
    df["sentiment_x_pagevalue"] = df["avg_sentiment_score"] * df.get("log_PageValues", df.get("PageValues", 0))
    # Engagement × ProductRatio (social engagement × on-site product interest)
    df["engagement_x_product_ratio"] = df["avg_engagement"] * df.get("product_page_ratio", 0)

    print(f"\n✅ Fusion complete. Final shape: {df.shape}")
    print(f"   Columns: {df.columns.tolist()}")
    return df.reset_index(drop=True)


df_fused = merge_data(df_ecommerce, df_social_agg, df_product)
print(f"\nRevenue distribution in fused dataset:")
print(df_fused["Revenue"].value_counts())
display(df_fused.head(3))


---
## 🤖 Section 5 – `train_model()`: Machine Learning


In [ ]:
# ── Feature selection ─────────────────────────────────────────────────────
FEATURE_COLS = [
    # ── Behavioral ─────────────────────────────────────────────────────
    "BounceRates", "ExitRates", "log_PageValues", "has_page_value",
    "log_ProductRelated_Duration", "product_page_ratio",
    "total_pages", "total_duration_sec",
    "SpecialDay", "Weekend", "VisitorType_enc", "Month_num",
    # ── Social Fusion ───────────────────────────────────────────────────
    "avg_sentiment_score", "pct_positive", "pct_negative",
    "avg_engagement", "std_sentiment_score", "total_tweets",
    "sentiment_x_pagevalue", "engagement_x_product_ratio",
    # ── Product Fusion ──────────────────────────────────────────────────
    "global_avg_price", "global_avg_stars", "global_bestseller_rate",
    "global_avg_discount_pct",
]

TARGET_COL = "Revenue"


def prepare_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """
    Select and validate feature columns.
    Returns X (features) and y (target).
    """
    available = [c for c in FEATURE_COLS if c in df.columns]
    missing   = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        print(f"   ⚠ Features not found (skipped): {missing}")

    X = df[available].copy()
    y = df[TARGET_COL].copy()

    # Ensure all numeric
    X = X.apply(pd.to_numeric, errors="coerce")
    X.fillna(X.median(), inplace=True)

    print(f"   Feature matrix shape : {X.shape}")
    print(f"   Target distribution  :\n{y.value_counts().to_string()}")
    return X, y


def train_model(df: pd.DataFrame) -> dict:
    """
    Train Logistic Regression + Random Forest on the fused dataset.

    Returns a results dict with trained models, test data, and metrics.
    """
    print("\n" + "=" * 65)
    print("  TRAIN MODEL")
    print("=" * 65)

    X, y = prepare_features(df)

    # ── Train/test split ──────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\n   Train size: {len(X_train):,} | Test size: {len(X_test):,}")

    # ── Pipelines (impute + scale for LR; just impute for RF) ────
    lr_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("clf",     LogisticRegression(
            max_iter=1000, random_state=RANDOM_STATE,
            class_weight="balanced", C=0.5
        )),
    ])

    rf_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf",     RandomForestClassifier(
            n_estimators=200, max_depth=12,
            random_state=RANDOM_STATE, class_weight="balanced",
            n_jobs=-1
        )),
    ])

    models = {
        "Logistic Regression": lr_pipe,
        "Random Forest":       rf_pipe,
    }

    results = {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "feature_names": X.columns.tolist(),
        "models": {},
    }

    for name, pipe in models.items():
        print(f"\n🔹 Training {name} …")
        pipe.fit(X_train, y_train)

        y_pred  = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]

        # 5-fold CV AUC
        cv_auc = cross_val_score(pipe, X_train, y_train, cv=5,
                                  scoring="roc_auc", n_jobs=-1)

        metrics = {
            "accuracy"  : accuracy_score(y_test, y_pred),
            "precision" : precision_score(y_test, y_pred, zero_division=0),
            "recall"    : recall_score(y_test, y_pred, zero_division=0),
            "f1"        : f1_score(y_test, y_pred, zero_division=0),
            "roc_auc"   : roc_auc_score(y_test, y_proba),
            "cv_auc_mean": cv_auc.mean(),
            "cv_auc_std" : cv_auc.std(),
        }

        print(f"   Accuracy   : {metrics['accuracy']:.4f}")
        print(f"   Precision  : {metrics['precision']:.4f}")
        print(f"   Recall     : {metrics['recall']:.4f}")
        print(f"   F1-Score   : {metrics['f1']:.4f}")
        print(f"   ROC-AUC    : {metrics['roc_auc']:.4f}")
        print(f"   CV ROC-AUC : {metrics['cv_auc_mean']:.4f} ± {metrics['cv_auc_std']:.4f}")

        results["models"][name] = {
            "pipeline": pipe,
            "metrics" : metrics,
            "y_pred"  : y_pred,
            "y_proba" : y_proba,
        }

    print("\n✅ Training complete.")
    return results


model_results = train_model(df_fused)


---
## 📊 Section 6 – `evaluate_model()`: Evaluation & Feature Importance


In [ ]:
def evaluate_model(results: dict) -> None:
    """
    Print classification reports, confusion matrices, ROC curves,
    and feature importance for all trained models.
    """
    print("\n" + "=" * 65)
    print("  MODEL EVALUATION")
    print("=" * 65)

    y_test = results["y_test"]
    feat_names = results["feature_names"]

    # ── Metrics summary table ─────────────────────────────────────
    rows = []
    for name, data in results["models"].items():
        m = data["metrics"]
        rows.append({
            "Model"     : name,
            "Accuracy"  : m["accuracy"],
            "Precision" : m["precision"],
            "Recall"    : m["recall"],
            "F1-Score"  : m["f1"],
            "ROC-AUC"   : m["roc_auc"],
            "CV AUC (mean±std)": f"{m['cv_auc_mean']:.4f} ± {m['cv_auc_std']:.4f}",
        })
    metrics_df = pd.DataFrame(rows).set_index("Model")
    print("\n📋 Model Performance Summary:")
    display(metrics_df)

    # ── Classification reports ────────────────────────────────────
    for name, data in results["models"].items():
        print(f"\n{'='*40}")
        print(f"  {name} — Classification Report")
        print(f"{'='*40}")
        print(classification_report(y_test, data["y_pred"],
                                     target_names=["No Purchase", "Purchase"]))

    # ── ROC Curves ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("ROC Curves – All Models", fontsize=14, fontweight="bold")

    for ax, (name, data) in zip(axes, results["models"].items()):
        fpr, tpr, _ = roc_curve(y_test, data["y_proba"])
        ax.plot(fpr, tpr, color=PALETTE[0], lw=2,
                label=f"AUC = {data['metrics']['roc_auc']:.4f}")
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
        ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE[0])
        ax.set_title(name)
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right")
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.02])

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "roc_curves.png", bbox_inches="tight")
    plt.show()

    # ── Confusion Matrices ────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")

    for ax, (name, data) in zip(axes, results["models"].items()):
        cm = confusion_matrix(y_test, data["y_pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["No Purchase", "Purchase"],
                    yticklabels=["No Purchase", "Purchase"],
                    linewidths=0.5, cbar=False)
        ax.set_title(name)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrices.png", bbox_inches="tight")
    plt.show()

    # ── Feature Importance (Random Forest) ───────────────────────
    if "Random Forest" in results["models"]:
        rf_data = results["models"]["Random Forest"]
        clf = rf_data["pipeline"].named_steps["clf"]
        importances = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(10, 7))
        colors = [PALETTE[0] if "sentiment" in f or "engagement" in f
                  else PALETTE[1] if "page" in f.lower() or "bounce" in f.lower()
                  else PALETTE[2] for f in importances.index[:20]]
        importances[:20].plot(kind="barh", ax=ax, color=colors[::-1])
        ax.invert_yaxis()
        ax.set_title("Top-20 Feature Importances (Random Forest)", fontweight="bold")
        ax.set_xlabel("Importance Score")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "feature_importance.png", bbox_inches="tight")
        plt.show()

        print("\n🏆 Top-10 Most Important Features:")
        display(importances.head(10).reset_index().rename(columns={"index": "Feature", 0: "Importance"}))

    print("\n✅ Evaluation complete.")


evaluate_model(model_results)


---
## 📈 Section 7 – `visualize()`: Insights & Analysis Plots


In [ ]:
def visualize(df_fused: pd.DataFrame, df_product: pd.DataFrame,
              df_twitter: pd.DataFrame) -> None:
    """
    Generate six key insight visualizations.
    """
    print("\n" + "=" * 65)
    print("  VISUALIZATIONS")
    print("=" * 65)

    rev_label = {0: "No Purchase", 1: "Purchase"}
    df_fused["Revenue_label"] = df_fused["Revenue"].map(rev_label)

    # ── 1. Sentiment vs Purchase ──────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("1. Monthly Sentiment vs Purchase Rate", fontsize=13, fontweight="bold")

    ax = axes[0]
    pivot = df_fused.groupby("Month_num").agg(
        avg_sentiment = ("avg_sentiment_score", "mean"),
        purchase_rate = ("Revenue", "mean"),
    )
    ax2 = ax.twinx()
    ax.bar(pivot.index, pivot["purchase_rate"] * 100, color=PALETTE[1], alpha=0.7, label="Purchase Rate %")
    ax2.plot(pivot.index, pivot["avg_sentiment"], color=PALETTE[0], marker="o", lw=2, label="Avg Sentiment")
    ax.set_xlabel("Month")
    ax.set_ylabel("Purchase Rate (%)", color=PALETTE[1])
    ax2.set_ylabel("Avg Sentiment Score", color=PALETTE[0])
    ax.set_title("Purchase Rate vs Avg Social Sentiment by Month")
    ax.set_xticks(range(1, 13))
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

    ax = axes[1]
    sent_rv = df_fused.groupby("Revenue_label")["avg_sentiment_score"].mean()
    bars = ax.bar(sent_rv.index, sent_rv.values, color=PALETTE[:2], edgecolor="white", width=0.5)
    for bar, val in zip(bars, sent_rv.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=11)
    ax.set_title("Average Monthly Sentiment Score by Purchase Outcome")
    ax.set_ylabel("Avg Sentiment Score")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sentiment_vs_purchase.png", bbox_inches="tight")
    plt.show()

    # ── 2. Engagement vs Purchase ─────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("2. Social Engagement vs Purchase Behavior", fontsize=13, fontweight="bold")

    ax = axes[0]
    eng_rv = df_fused.groupby("Revenue_label")["avg_engagement"].mean()
    bars = ax.bar(eng_rv.index, eng_rv.values, color=PALETTE[2:4], edgecolor="white", width=0.5)
    for bar, val in zip(bars, eng_rv.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=11)
    ax.set_title("Avg Social Engagement by Purchase Outcome")
    ax.set_ylabel("Avg Engagement (Likes + Retweets)")

    ax = axes[1]
    df_sample = df_fused.sample(min(5000, len(df_fused)), random_state=42)
    for rev_val, label, color in [(0, "No Purchase", PALETTE[2]), (1, "Purchase", PALETTE[3])]:
        sub = df_sample[df_sample["Revenue"] == rev_val]
        ax.scatter(sub["avg_engagement"], sub["log_PageValues"],
                   c=color, label=label, alpha=0.4, s=15)
    ax.set_xlabel("Avg Social Engagement")
    ax.set_ylabel("Log(Page Value)")
    ax.set_title("Engagement × Page Value by Outcome")
    ax.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "engagement_vs_purchase.png", bbox_inches="tight")
    plt.show()

    # ── 3. Price vs Purchase (Product Dataset) ────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("3. Product Price Analysis", fontsize=13, fontweight="bold")

    ax = axes[0]
    cat_price = df_product.groupby("category_broad")["price"].median().sort_values(ascending=False).head(12)
    ax.barh(cat_price.index, cat_price.values, color=PALETTE[4], edgecolor="white")
    ax.set_xlabel("Median Price (BRL)")
    ax.set_title("Median Product Price by Category")
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}"))

    ax = axes[1]
    ax.hist(np.log1p(df_product["price"]), bins=50, color=PALETTE[0], edgecolor="none", alpha=0.8)
    ax.set_xlabel("Log(Price + 1)")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of Product Prices (log-scale)")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "price_analysis.png", bbox_inches="tight")
    plt.show()

    # ── 4. Category vs Purchase (ecommerce proxy) ─────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("4. Product Catalog – Category & Popularity", fontsize=13, fontweight="bold")

    ax = axes[0]
    cat_counts = df_product["category_broad"].value_counts().head(12)
    ax.bar(range(len(cat_counts)), cat_counts.values,
           color=sns.color_palette("husl", len(cat_counts)), edgecolor="none")
    ax.set_xticks(range(len(cat_counts)))
    ax.set_xticklabels(cat_counts.index, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Product Count")
    ax.set_title("Products per Broad Category")

    ax = axes[1]
    bestseller = df_product.groupby("category_broad")["isBestSeller"].mean().sort_values().tail(10)
    ax.barh(bestseller.index, bestseller.values * 100, color=PALETTE[1], edgecolor="white")
    ax.set_xlabel("Best-Seller Rate (%)")
    ax.set_title("Best-Seller Rate by Category")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "category_analysis.png", bbox_inches="tight")
    plt.show()

    # ── 5. Correlation Heatmap ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 10))
    heat_cols = [c for c in [
        "Revenue", "avg_sentiment_score", "pct_positive", "pct_negative",
        "avg_engagement", "std_sentiment_score",
        "BounceRates", "ExitRates", "log_PageValues", "has_page_value",
        "product_page_ratio", "total_duration_sec",
        "global_avg_price", "global_avg_stars",
        "sentiment_x_pagevalue", "engagement_x_product_ratio",
        "SpecialDay", "Weekend",
    ] if c in df_fused.columns]

    corr = df_fused[heat_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
                center=0, linewidths=0.5, ax=ax,
                annot_kws={"size": 7}, vmin=-1, vmax=1)
    ax.set_title("Correlation Heatmap – Fused Feature Space", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", bbox_inches="tight")
    plt.show()

    # ── 6. Twitter Sentiment Distribution ────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("5. Social Media Sentiment Distribution", fontsize=13, fontweight="bold")

    ax = axes[0]
    sent_counts = df_twitter["sentiment"].value_counts()
    ax.pie(sent_counts.values, labels=sent_counts.index,
           colors=PALETTE[:3], autopct="%1.1f%%",
           startangle=90, wedgeprops=dict(edgecolor="white"))
    ax.set_title("Sentiment Label Split")

    ax = axes[1]
    ax.hist(df_twitter["sentiment_score"], bins=50, color=PALETTE[0], edgecolor="none", alpha=0.8)
    ax.axvline(0, color="white", lw=1.5, linestyle="--", alpha=0.6)
    ax.set_xlabel("Sentiment Score")
    ax.set_ylabel("Tweet Count")
    ax.set_title("Sentiment Score Distribution")

    ax = axes[2]
    df_twitter["engagement"] = df_twitter["like_count"] + df_twitter["retweet_count"]
    sent_eng = df_twitter.groupby("sentiment")["engagement"].mean().sort_values()
    ax.barh(sent_eng.index, sent_eng.values, color=PALETTE[:3], edgecolor="white")
    ax.set_xlabel("Avg Engagement (Likes + Retweets)")
    ax.set_title("Engagement by Sentiment Label")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sentiment_distribution.png", bbox_inches="tight")
    plt.show()

    print("\n✅ All visualizations saved to:", OUTPUT_DIR)


visualize(df_fused, df_product, df_twitter)


---
## 💡 Section 8 – Key Insights & Research Questions


In [ ]:
def generate_insights(df_fused: pd.DataFrame, model_results: dict) -> None:
    """
    Answer the four core research questions from the project objective.
    """
    print("=" * 65)
    print("  KEY INSIGHTS — Data Fusion Analysis")
    print("=" * 65)

    rev_rate = df_fused["Revenue"].mean() * 100

    # ── Q1: Does sentiment affect purchase? ───────────────────────
    sent_by_rev = df_fused.groupby("Revenue")["avg_sentiment_score"].mean()
    delta_sent  = sent_by_rev[1] - sent_by_rev[0]

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Q1 ▸ Does social media SENTIMENT affect purchase conversion?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Overall conversion rate   : {rev_rate:.2f}%
   Avg sentiment – No Purchase: {sent_by_rev[0]:+.4f}
   Avg sentiment – Purchase   : {sent_by_rev[1]:+.4f}
   Delta                      : {delta_sent:+.4f}

   ➜ {'Higher positive sentiment correlates with MORE purchases.' if delta_sent > 0 else 'No clear positive sentiment lift detected — engagement matters more.'}
    """)

    # ── Q2: Does engagement matter? ───────────────────────────────
    eng_by_rev = df_fused.groupby("Revenue")["avg_engagement"].mean()
    delta_eng  = eng_by_rev[1] - eng_by_rev[0]
    pct_lift   = (delta_eng / eng_by_rev[0]) * 100

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Q2 ▸ Does social media ENGAGEMENT matter?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Avg engagement – No Purchase: {eng_by_rev[0]:.4f}
   Avg engagement – Purchase   : {eng_by_rev[1]:.4f}
   Lift                        : {delta_eng:+.4f} ({pct_lift:+.1f}%)

   ➜ {'Higher social engagement months show higher purchase rates — buzz drives traffic.' if delta_eng > 0 else 'Engagement alone is not a strong differentiator; content quality matters.'}
    """)

    # ── Q3: Does price reduce conversion? ─────────────────────────
    price_ref = df_fused["global_avg_price"].iloc[0]
    pv_by_rev = df_fused.groupby("Revenue")["log_PageValues"].mean()

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Q3 ▸ Does PRICE reduce conversion?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Global avg product price (catalog): R$ {price_ref:.2f}
   Log PageValue – No Purchase        : {pv_by_rev[0]:.4f}
   Log PageValue – Purchase           : {pv_by_rev[1]:.4f}

   ➜ Page value (a proxy for price × engagement) is significantly
     higher for purchasers. While higher catalog prices may deter
     some, value-for-money signals (discounts, best-sellers) boost
     conversion. Price tier moderation matters.
    """)

    # ── Q4: Most important feature? ───────────────────────────────
    if "Random Forest" in model_results["models"]:
        feat_names = model_results["feature_names"]
        clf = model_results["models"]["Random Forest"]["pipeline"].named_steps["clf"]
        importances = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)
        top_feat  = importances.index[0]
        top_score = importances.iloc[0]

        social_feats = [f for f in importances.index if any(
            kw in f for kw in ["sentiment", "engagement", "pct_pos", "pct_neg"])]
        social_total = importances[social_feats].sum()

        print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Q4 ▸ Which factor is MOST IMPORTANT for predicting purchase?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   #1 Feature   : {top_feat} ({top_score:.4f})
   Social total : {social_total:.4f} combined importance

   Top-5 features:
{importances.head(5).to_string()}

   ➜ On-site behavioral signals (PageValues, BounceRates) are the
     strongest predictors. Social media features contribute a measurable
     but secondary effect — they amplify the signal, particularly
     via the sentiment × page_value interaction feature.
        """)

    print("\n✅ Insight generation complete.")


generate_insights(df_fused, model_results)


---
## 💾 Section 9 – Save Outputs (CSV + SQLite)


In [ ]:
def save_outputs(df_fused: pd.DataFrame, model_results: dict) -> None:
    """
    Persist the final fused dataset (CSV) and model metrics (SQLite + CSV).
    """
    print("\n" + "=" * 65)
    print("  SAVE OUTPUTS")
    print("=" * 65)

    # ── 1. Final dataset → CSV ────────────────────────────────────
    csv_path = OUTPUT_DIR / "final_fused_dataset.csv"
    df_fused.to_csv(csv_path, index=False)
    print(f"   ✅ Fused dataset saved   : {csv_path}")
    print(f"      Shape : {df_fused.shape}")

    # ── 2. Model metrics → CSV ───────────────────────────────────
    rows = []
    for name, data in model_results["models"].items():
        m = data["metrics"]
        row = {"model": name}
        row.update(m)
        rows.append(row)
    metrics_df = pd.DataFrame(rows)
    metrics_csv = OUTPUT_DIR / "model_metrics.csv"
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"   ✅ Model metrics saved   : {metrics_csv}")

    # ── 3. SQLite ─────────────────────────────────────────────────
    db_path = OUTPUT_DIR / "data_fusion.db"
    con = sqlite3.connect(db_path)

    # Store a 50k-row sample (SQLite keeps file manageable)
    sample = df_fused.sample(min(50_000, len(df_fused)), random_state=42)
    sample.to_sql("fused_sessions",  con, if_exists="replace", index=False)
    metrics_df.to_sql("model_metrics", con, if_exists="replace", index=False)

    # Quick validation query
    count = pd.read_sql("SELECT COUNT(*) AS n FROM fused_sessions", con).iloc[0, 0]
    rev   = pd.read_sql("SELECT Revenue, COUNT(*) AS cnt FROM fused_sessions GROUP BY Revenue", con)
    con.close()

    print(f"   ✅ SQLite database saved : {db_path}")
    print(f"      Rows in DB           : {count:,}")
    print(f"      Revenue split in DB  :\n{rev.to_string(index=False)}")

    print("\n📁 Output files:")
    for f in sorted(OUTPUT_DIR.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"   {f.name:<40} {size_kb:>8.1f} KB")


save_outputs(df_fused, model_results)


---
## 🚀 Section 10 – `pipeline()`: Full End-to-End Execution

Run the entire pipeline in one call — useful for automation or CI.


In [ ]:
def pipeline(
    product_sample: int | None = 300_000,
    ecommerce_sample: int | None = 300_000,
    twitter_sample: int | None = 200_000,
) -> dict:
    """
    Complete end-to-end Data Fusion Pipeline.

    Stages
    ------
    1. Load raw data
    2. Clean each dataset
    3. Feature engineering
    4. Aggregate social media
    5. Merge / fuse datasets
    6. Train ML models
    7. Evaluate models
    8. Visualize insights
    9. Save outputs

    Returns
    -------
    dict with 'df_fused' and 'model_results'
    """
    import time
    t0 = time.time()

    print("🚀 Starting Data Fusion Pipeline …")
    print(f"   Samples: product={product_sample:,} | ecommerce={ecommerce_sample:,} | twitter={twitter_sample:,}")

    # Stage 1
    df_product_raw, df_ecommerce_raw, df_twitter_raw = load_data(
        product_sample=product_sample,
        ecommerce_sample=ecommerce_sample,
        twitter_sample=twitter_sample,
    )

    # Stage 2
    df_product_c  = clean_product(df_product_raw)
    df_ecommerce_c = clean_ecommerce(df_ecommerce_raw)
    df_twitter_c  = clean_social(df_twitter_raw)

    # Stage 3
    df_product_f  = feature_engineering_product(df_product_c)
    df_ecommerce_f = feature_engineering_ecommerce(df_ecommerce_c)

    # Stage 4
    social_agg = aggregate_social(df_twitter_c)

    # Stage 5
    fused = merge_data(df_ecommerce_f, social_agg, df_product_f)

    # Stage 6
    results = train_model(fused)

    # Stage 7
    evaluate_model(results)

    # Stage 8
    visualize(fused, df_product_f, df_twitter_c)

    # Stage 9 (insights + save)
    generate_insights(fused, results)
    save_outputs(fused, results)

    elapsed = time.time() - t0
    print(f"\n🏁 Pipeline complete in {elapsed:.1f}s")
    return {"df_fused": fused, "model_results": results}


# ── Uncomment the line below to run the full pipeline from scratch ──────────
# pipeline_output = pipeline()
print("✅ pipeline() function defined. Call pipeline() to run end-to-end.")


---
## 📋 Project Summary

| Component | Detail |
|-----------|--------|
| **Datasets** | Amazon BR Products (~1.34M), Online Shoppers (~1M), Twitter Sentiment (~503K) |
| **Fusion Strategy** | Monthly social sentiment joined to ecommerce sessions; product catalog statistics broadcast |
| **Models** | Logistic Regression + Random Forest (class-balanced, cross-validated) |
| **Target** | `Revenue` — binary purchase conversion |
| **Key Fusion Features** | `avg_sentiment_score`, `pct_positive`, `avg_engagement`, `sentiment_x_pagevalue` |
| **Outputs** | `output/final_fused_dataset.csv`, `output/data_fusion.db`, `output/model_metrics.csv`, 6 PNG plots |

### Research Conclusions
1. **Sentiment** — monthly social sentiment has a measurable positive correlation with conversion rate
2. **Engagement** — months with higher social engagement tend to have higher purchase rates
3. **Price** — on-site page value (price × engagement proxy) is the #1 behavioral predictor; catalog price diversity shows that budget/mid-tier items convert best
4. **Most Important Factor** — on-site behavioral signals (PageValues, BounceRates) dominate; social features provide meaningful secondary lift via the fused interaction terms
